# Pinch Analysis: Crude Preheat Train
This notebook shows how the pinch, utility targets, and graph shapes move when the minimum approach temperature assumption changes.


## Case context
The packaged `crude_preheat_train.json` case represents a small crude-unit preheat train with named hot and cold duties. The workflow is: solve the baseline case, inspect the main graphs, then rerun the case by changing the root-zone `dt_cont_multiplier`, which scales every process and utility stream `dt_cont` value through the zone hierarchy.


In [26]:
from copy import deepcopy
import json
import numpy as np
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from OpenPinch import PinchProblem
from OpenPinch.resources import copy_sample_case

In [27]:
workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
case_path = copy_sample_case(
    "crude_preheat_train.json",
    work_dir / "crude_preheat_train.json",
)

In [28]:
problem = PinchProblem(source=case_path)
problem.target()
summary = problem.summary_frame()
base_row = summary.loc[
    summary["Target"] == "crude_preheat_train/Direct Integration",
    [
        "Hot Utility Target",
        "Cold Utility Target",
        "Heat Recovery",
        "Hot Pinch",
        "Cold Pinch",
    ],
]
print(summary)

                          Target  Hot Utility Target  Cold Utility Target  \
0        Site/Direct Integration          749.999995               1000.0   
1      Site/Total Process Target          749.999995               1000.0   
2         Site/Total Site Target          749.999995               1000.0   
3  Crude Unit/Direct Integration          749.999995               1000.0   

   Heat Recovery  Hot Pinch  Cold Pinch     Hot Utilities  \
0    5150.000005      145.0       145.0  MP Steam: 750.00   
1    5150.000005        NaN         NaN  MP Steam: 750.00   
2    5150.000005      258.9        11.1  MP Steam: 750.00   
3    5150.000005      145.0       145.0  MP Steam: 750.00   

           Cold Utilities  
0  Cooling Water: 1000.00  
1  Cooling Water: 1000.00  
2  Cooling Water: 1000.00  
3  Cooling Water: 1000.00  


## Baseline graphs
Render the baseline composite curve, shifted composite curve, and grand composite curve directly in the notebook so the graph shapes can be read before changing `dt_cont`.


In [29]:
problem.graph_catalog()

,Zone,Zone Address,Target,Target Type,Graph Type,Graph Name,Index
0,Site,Site,Site/Direct Integration,Direct Integration,Composite Curves,Graph 1,0
1,Site,Site,Site/Direct Integration,Direct Integration,Shifted Composite Curves,Graph 2,1
2,Site,Site,Site/Direct Integration,Direct Integration,Balanced Composite Curves,Graph 3,2
3,Site,Site,Site/Direct Integration,Direct Integration,Grand Composite Curve,Graph 4,3
4,Site,Site,Site/Direct Integration,Direct Integration,Net Load Profiles,Graph 5,4
5,Site,Site,Site/Direct Integration,Direct Integration,Grand Composite Curve with Heat Pump,Graph 6,5
6,Site,Site,Site/Total Site Target,Total Site Target,Total Site Profiles,Graph 1,0
7,Site,Site,Site/Total Site Target,Total Site Target,Site Utility Grand Composite Curve,Graph 2,1
8,Crude Unit,Site/Crude Unit,Crude Unit/Direct Integration,Direct Integration,Composite Curves,Graph 1,0
9,Crude Unit,Site/Crude Unit,Crude Unit/Direct Integration,Direct Integration,Shifted Composite Curves,Graph 2,1


In [30]:
cc = problem.plot.composite_curve(show=True)

In [31]:
scc = problem.plot.shifted_composite_curve(show=True)

In [32]:
bcc = problem.plot.balanced_composite_curve(show=True)

In [33]:
gcc = problem.plot.grand_composite_curve(show=True)

In [34]:
nlc = problem.plot.net_load_profiles(show=True)

In [35]:
multipliers = np.linspace(0.5, 10.0, 26)

problem = PinchProblem(source=case_path, project_name=case_path.stem)

rows = []
for factor in multipliers:
    problem.set_dt_cont_multiplier(factor)
    summary = problem.summary_frame()
    row = summary.loc[summary["Target"] == f"{case_path.stem}/Direct Integration"].iloc[
        0
    ]
    rows.append(
        {
            "dt_cont multiplier": factor,
            "Hot Utility Target": row["Hot Utility Target"],
            "Cold Utility Target": row["Cold Utility Target"],
            "Heat Recovery": row["Heat Recovery"],
            "Hot Pinch": row["Hot Pinch"],
            "Cold Pinch": row["Cold Pinch"],
        }
    )

sensitivity = pd.DataFrame(rows)
sensitivity

,dt_cont multiplier,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch
0,0.50,549.999995,800.0,5350.000005,142.5,142.5
1,0.88,701.999995,952.0,5198.000005,144.4,144.4
2,1.26,853.999995,1104.0,5046.000005,146.3,146.3
3,1.64,1005.999995,1256.0,4894.000005,148.2,148.2
4,2.02,1157.999995,1408.0,4742.000005,150.1,150.1
5,2.40,1309.999995,1560.0,4590.000005,152.0,152.0
6,2.78,1461.999995,1712.0,4438.000005,153.9,153.9
7,3.16,1613.999995,1864.0,4286.000005,155.8,155.8
8,3.54,1765.999995,2016.0,4134.000005,157.7,157.7
9,3.92,1917.999995,2168.0,3982.000005,159.6,159.6


In [36]:
series_colors = {
    "Hot Utility Target": "#c0392b",
    "Cold Utility Target": "#2471a3",
    "Heat Recovery": "#138d75",
    "Hot Pinch": "#d68910",
    "Cold Pinch": "#7d3c98",
}

sensitivity_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Utility Targets and Heat Recovery",
        "Pinch Temperatures",
    ),
)

for column in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=1,
        col=1,
    )

for column in ["Hot Pinch", "Cold Pinch"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=2,
        col=1,
    )

sensitivity_fig.update_xaxes(title_text="dt_cont multiplier", row=2, col=1)
sensitivity_fig.update_yaxes(title_text="kW or degC", row=1, col=1)
sensitivity_fig.update_yaxes(title_text="degC", row=2, col=1)
sensitivity_fig.update_layout(title="dt_cont sensitivity overview")
# display_plotly(sensitivity_fig, height=700)